# 五天通关回顾 · 结营本

这本是**结营用的总复习本**：带着你把 Day1→Day5 亲手再走一遍。

目标不是再学新算法，而是：

```text
1. 回忆：这五天各自解决什么问题
2. 动手：用同一份咖啡馆数据，再跑通整条链
3. 展示：能指着结果说一句「发现 + 建议」
```

数据文件：同目录 `day5_cafe_sales.csv`
演示网页（可选）：`python mini_predict_app.py` → `http://127.0.0.1:5001`


## 0. 五天地图（先齐读）

| 天 | 一句话 | 你学会的「武器」 |
| --- | --- | --- |
| Day 1 | 会用 Python 摸数据 | 变量、循环感觉；读 CSV；看表 |
| Day 2 | 会把表切成经营场景 | 选列、筛选、排序、groupby |
| Day 3 | 会训模型猜营收 | 特征/目标；决策树→随机森林；重要性；predict |
| Day 4 | 会判断模型好不好 | R² / MAE / RMSE；多模型对比；调参感觉；残差 |
| Day 5 | 会把结果讲给人听 | 场景证据 + what-if + 迷你网页演示 |

整条工作流：

```text
提问 → 取证（表） → 模型辅助 → 做成能演示的成果 → 建议与验证
```


## 0.5 先把环境跑通

下面一格：导入库 + 读表。**跑通再往下。**


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# 中文图（乱码可忽略，不影响数字）
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

candidate_paths = [
    Path('day5_cafe_sales.csv'),
    Path('day5') / 'day5_cafe_sales.csv',
    Path('教学课程') / 'day5' / 'day5_cafe_sales.csv',
]
for path in candidate_paths:
    if path.exists():
        csv_path = path
        break
else:
    raise FileNotFoundError('未找到 day5_cafe_sales.csv，请把工作目录切到 day5')

df_raw = pd.read_csv(csv_path)
print('路径:', csv_path.resolve())
print('shape:', df_raw.shape)
print('列名:', list(df_raw.columns))
df_raw.head()


---

# 通关 1 · Day 1 回忆：Python 与数据感觉

Day 1 解决的问题：

> 电脑语言不再吓人；能打开一张表，知道有多少天、营收大概多少。

### 口头 30 秒

1. 变量像不像「贴了名字的盒子」？
2. `df.head()` 是干什么的？
3. 为什么要先看表，再分析？


In [ ]:
# Day1 技能复刻：基础统计
df = df_raw.copy()

n_days = len(df)
mean_sales = df['sales'].mean()
max_sales = df['sales'].max()
min_sales = df['sales'].min()

print(f'一共 {n_days} 天')
print(f'平均营收 ≈ {mean_sales:.1f} 元')
print(f'最高 / 最低 ≈ {max_sales:.1f} / {min_sales:.1f} 元')

# 小挑战：中位数
print('中位数 ≈', df['sales'].median())


In [ ]:
# 可选：营收走势一眼（Day1 图的感觉）
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(df['day'], df['sales'], color='#6f4e37', linewidth=1.2)
ax.set_title('营收走势（Day1 感觉）')
ax.set_xlabel('day')
ax.set_ylabel('sales')
plt.tight_layout()
plt.show()


### 通关印章 · Day1

在下面格子里，用**自己的话**写一句：

```text
Day1 我带走的是：________
```


In [ ]:
# Day1 我带走的是：
#


---

# 通关 2 · Day 2 回忆：把表切成经营场景

Day 2 解决的问题：

> 不只看「平均多少」，而会问：晴天怎样？周末怎样？有促销怎样？

常用招式：

| 招式 | 干什么 |
| --- | --- |
| 选列 | 只要关心的字段 |
| 筛选 | 只要满足条件的行 |
| 排序 | 找 TOP / BOTTOM |
| groupby | 按场景算平均 |


In [ ]:
# Day2 技能复刻：打标签 + 场景对比
df = df_raw.copy()
df['weekend_label'] = df['is_weekend'].map({0: '工作日', 1: '周末'})
df['promo_label'] = df['promotion'].map({0: '无促销', 1: '有促销'})

print('=== 按天气 ===')
print(df.groupby('weather_label')['sales'].mean().round(1).sort_values(ascending=False))
print()
print('=== 按周末 ===')
print(df.groupby('weekend_label')['sales'].mean().round(1))
print()
print('=== 按促销 ===')
print(df.groupby('promo_label')['sales'].mean().round(1))


In [ ]:
# 交叉表：周末 × 促销（经营故事常用）
pivot = df.pivot_table(
    values='sales',
    index='weekend_label',
    columns='promo_label',
    aggfunc='mean',
).round(1)
print(pivot)

# 筛选例子：只要「有促销的周末」
mask = (df['is_weekend'] == 1) & (df['promotion'] == 1)
print()
print('有促销的周末天数:', int(mask.sum()))
print('这些天平均营收:', round(df.loc[mask, 'sales'].mean(), 1))


In [ ]:
# 小挑战：TOP5 与 BOTTOM5 营收日
print('=== TOP5 ===')
print(df.nlargest(5, 'sales')[['day', 'weather_label', 'weekend_label', 'promo_label', 'sales']])
print()
print('=== BOTTOM5 ===')
print(df.nsmallest(5, 'sales')[['day', 'weather_label', 'weekend_label', 'promo_label', 'sales']])


### 发现 → 解释 → 行动（写 2 条）

模板：

```text
发现：……（带数字）
解释：……（为什么可能这样）
行动：……（店里可以试什么 / 怎么验证）
```


In [ ]:
# 发现 1：
# 解释 1：
# 行动 1：
#
# 发现 2：
# 解释 2：
# 行动 2：


---

# 通关 3 · Day 3 回忆：从规则到森林

Day 3 解决的问题：

> 不只描述历史，还能**用特征猜新一天的营收**。

### 和最简单模型对上号

```text
Y = KX + B
```

- **X**：输入（价格、促销、天气……）
- **Y**：输出（营收）
- 训练：让预测和真实差得少一点（损失变小）
- 评估：换新数据看还灵不灵

| 模型 | 直觉 |
| --- | --- |
| 直线 `Y=KX+B` | 一条线去拟合 |
| 决策树 | 一串 if/else 去切分 |
| 随机森林 | 很多棵树一起平均，通常更稳 |

**特征 X** = 用来判断的列；**目标 y** = 要预测的 `sales`。
训练集用来学，测试集用来考——避免「自己考自己满分」（过拟合）。


In [ ]:
# Day3 技能复刻：特征工程一点点 + 训随机森林
df = df_raw.copy()
weather_score_map = {'晴': 1.0, '多云': 0.8, '阴': 0.6, '小雨': 0.4, '大雨': 0.3}
df['weather_score'] = df['weather_label'].map(weather_score_map)

feature_cols = [
    'price', 'promotion', 'is_weekend', 'temp', 'quality',
    'competitors', 'holiday', 'location', 'weather_score',
]
X = df[feature_cols]
y = df['sales']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(
    n_estimators=100, max_depth=8, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

print('测试 R² :', round(r2_score(y_test, pred), 3))
print('测试 MAE:', round(mean_absolute_error(y_test, pred), 1))
print('训练天数 / 测试天数:', len(X_train), '/', len(X_test))


In [ ]:
# 特征重要性 → 业务话
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(imp.round(3))
print()
print('Top3:', list(imp.head(3).index))
print()
print('提醒：重要性高 = 模型更依赖它做预测，不等于严格因果。')


### 通关印章 · Day3

用业务话写 Top3（各半句）：


In [ ]:
# 1.
# 2.
# 3.
# 重要性 ≠ 因果，因为：


---

# 通关 4 · Day 4 回忆：打分、对比、别被一个数骗

Day 4 解决的问题：

> 模型训出来了，怎么知道好不好？换一个算法会不会更好？

| 指标 | 白话 |
| --- | --- |
| **R²** | 大概解释了多少波动（不是「准确率%」） |
| **MAE** | 平均偏了多少钱 |
| **RMSE** | 对大误差更敏感 |

项目里我们常**默认带随机森林**：可读重要性、方便 what-if；
**但真实数据上线性回归的 R² 有时更高**——要诚实读表，不要改种子刷名次。


In [ ]:
# Day4 技能复刻：多模型对比（同一划分）
models = {
    'LinearRegression': LinearRegression(),
    'DecisionTree(depth=8)': DecisionTreeRegressor(max_depth=8, random_state=42),
    'RandomForest(depth=8)': RandomForestRegressor(
        n_estimators=100, max_depth=8, random_state=42, n_jobs=-1
    ),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    p = model.predict(X_test)
    rows.append({
        'model': name,
        'R2': round(r2_score(y_test, p), 3),
        'MAE': round(mean_absolute_error(y_test, p), 1),
        'RMSE': round(mean_squared_error(y_test, p) ** 0.5, 1),
    })

compare = pd.DataFrame(rows).sort_values('R2', ascending=False)
print(compare.to_string(index=False))
print()
print('读表：谁 R² 高？谁 MAE 小？我们项目默认 RF 的理由是什么？')


In [ ]:
# 残差一眼：真实 - 预测，分布大概什么样
resid = y_test - rf.predict(X_test)
print('残差 mean ≈', round(float(resid.mean()), 1))
print('残差 std  ≈', round(float(resid.std()), 1))

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(resid, bins=15, color='#8b6914', edgecolor='white')
ax.set_title('测试集残差分布（Day4 感觉）')
ax.set_xlabel('y_true - y_pred')
plt.tight_layout()
plt.show()


### 口头讨论

1. R² = 0.76 能跟老板怎么说？（别说「准确率 76%」）
2. 如果线性回归 R² 更高，为什么项目仍可能带 RF 做经营诊断？
3. 残差特别大的天，你想回去查什么字段？


In [ ]:
# Day4 我带走的是：
#


---

# 通关 5 · Day 5 回忆：证据 + 能点的演示

Day 5 解决的问题：

> 不能只甩表格和 R²。要有**发现**，最好还能**拧旋钮给人看**。

迷你网页 `mini_predict_app.py` 的价值：

```text
同一套特征 → 点「训练」→ 改框 → 点「预测」
= 把 notebook 里的 what-if 变成老板能点的界面
```

它不是从零 Web 课；完整 `system/` 有余力再逛。


In [ ]:
# Day5 技能复刻：what-if（网页的预演 / 备用演示）
base = {
    'price': 25.0, 'promotion': 0, 'is_weekend': 0, 'temp': 22.0,
    'quality': 7.5, 'competitors': 2, 'holiday': 0, 'location': 7.0,
    'weather_score': 1.0,
}

def predict_one(overrides=None):
    row = base.copy()
    if overrides:
        row.update(overrides)
    x = pd.DataFrame([row], columns=feature_cols)
    return round(float(rf.predict(x)[0]), 1)

scenarios = {
    '基准': {},
    '有促销': {'promotion': 1},
    '周末': {'is_weekend': 1},
    '竞品+2': {'competitors': 4},
    '大雨': {'weather_score': 0.3},
    '单价+5': {'price': 30.0},
    '促销+周末': {'promotion': 1, 'is_weekend': 1},
}

print(f"{'场景':<12}{'预测营收':>10}")
print('-' * 24)
for name, ov in scenarios.items():
    print(f'{name:<12}{predict_one(ov):>10.1f}')

print()
print('规则：一次主要改一个旋钮，再解释「为什么可能变」。')
print('what-if ≠ 明天准账，是用来比较方案、排优先级。')


### 网页检查清单（若已启动）

在浏览器打开 `http://127.0.0.1:5001`：

1. 点「训练随机森林」→ 看到 R² / MAE / Top 特征
2. 只把 `promotion` 从 0 改 1 → 预测
3. 对照上表：数量级接近即可

启动命令（终端，工作目录 day5）：

```text
conda activate day1_ml
python mini_predict_app.py
```


---

# 终极大关 · 一条龙（五天压缩成 3 分钟）

下面一格是「闭卷提示」版：你先想步骤，再运行对照。


In [ ]:
# 终极大关：请按注释顺序在脑子里走一遍，再运行
# 1) 读表  2) weather_score  3) 切分  4) 训 RF  5) 打分  6) 重要性  7) 一个 what-if

df = df_raw.copy()
df['weather_score'] = df['weather_label'].map(
    {'晴': 1.0, '多云': 0.8, '阴': 0.6, '小雨': 0.4, '大雨': 0.3}
)
feature_cols = [
    'price', 'promotion', 'is_weekend', 'temp', 'quality',
    'competitors', 'holiday', 'location', 'weather_score',
]
X = df[feature_cols]
y = df['sales']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = RandomForestRegressor(
    n_estimators=100, max_depth=8, random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)
p = model.predict(X_test)

print('【评估】R² =', round(r2_score(y_test, p), 3),
      ' MAE =', round(mean_absolute_error(y_test, p), 1))
print('【Top3】', list(
    pd.Series(model.feature_importances_, index=feature_cols)
    .sort_values(ascending=False).head(3).index
))

demo = pd.DataFrame([{
    'price': 25, 'promotion': 1, 'is_weekend': 1, 'temp': 22,
    'quality': 8, 'competitors': 2, 'holiday': 0, 'location': 7,
    'weather_score': 1.0,
}], columns=feature_cols)
print('【演示】周末+促销 预测 ≈', round(float(model.predict(demo)[0]), 1), '元')
print()
print('一条龙过关')


---

# 结营 Show · 每人 60～90 秒

不要求念长稿。按这个骨架说即可：

```text
1. 一个带数字的发现（场景或模型）
2. 我拧了哪个旋钮 / 网页上点了什么
3. 给店里的一条建议（以及怎么验证）
```

### 可选加分句

- 提到 R² / MAE 时用**白话**
- 主动说一句局限（重要性≠因果 / what-if≠准账）
- 用一句话串五天


In [ ]:
# 我的 60 秒稿（提纲）
# 发现：
# 旋钮/演示：
# 建议：
# 局限：
# 五天一句话：


## 自我合格线（勾选）


In [ ]:
checklist = [
    '能读 CSV，说出天数和平均营收量级',
    '会 groupby / 筛选，讲出至少 2 条带数字场景发现',
    '能说出特征 X、目标 y，以及训练/测试为什么要分开',
    '能训练随机森林，并读出 R²、MAE（白话）',
    '能把 Top 特征重要性说成业务话，并知道≠因果',
    '会做至少一组 what-if，或能在迷你网页上演示',
    '能在 60～90 秒内讲清：发现 + 演示 + 建议',
    '能用一句话串起五天',
]
print('=== 结营自检 ===')
for i, t in enumerate(checklist, 1):
    print(f'{i}. [ ] {t}')
print()
print('勾完不代表完美，代表「这五天的主线你站得住」。')


## 五天链条（再齐读一遍）

```text
Day1  Python 与数据感觉
Day2  把表切成经营场景
Day3  树与森林：从特征猜营收
Day4  评估、对比、调参、残差
Day5  证据 + 能演示的成果 + 结营
```

工作流带走：

```text
提问 → 取证 → 模型辅助 → 演示 → 建议与验证
```

换奶茶店、换便利店、换自己学校的活动数据——链条一样用。

---

## 结营寄语

技术栈是手段。
**让听的人知道下一步做什么**，才是这五天的终点。

祝贺完成本次营期。
